In [ ]:
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

GAME = "antichess"
RUN_VARIANT = "oneshot"
LLM_MODEL = "openai-codex/gpt-5.5:xhigh"
TIMEOUT_SECONDS = 1800 if RUN_VARIANT == "agentic" else 900

USE_OPEN_SPIEL_BACKBONE = True
USE_IMPLEMENTATION_BRIEF = True

ROLLOUTS = 1000
MAX_STEPS = 1000
CHECK_SEED = 1

OUTPUT_DIR = Path("outputs")
PROMPT_PATH = Path("prompts/rulebook_to_python.txt")
BACKBONE_PATH = Path("prompts/open_spiel_backbone.md")
LLM_JUDGE_PROMPT_PATH = Path("prompts/llm_judge_review.md")
IMPLEMENTATION_BRIEF_PATH = OUTPUT_DIR / f"{GAME}_implementation_brief.md"

VARIANT_STEMS = {
    "oneshot": f"{GAME}_oneshot",
    "agentic": f"{GAME}_agentic",
}
if RUN_VARIANT not in VARIANT_STEMS:
    raise ValueError(f"Unsupported RUN_VARIANT: {RUN_VARIANT}")

RUN_STEM = VARIANT_STEMS[RUN_VARIANT]
CODE_PATH = OUTPUT_DIR / f"{RUN_STEM}.py"
RESPONSE_PATH = OUTPUT_DIR / f"{RUN_STEM}.md"
CHECK_LOG_PATH = OUTPUT_DIR / f"{RUN_STEM}_checks.txt"
JUDGE_PACKET_PATH = OUTPUT_DIR / f"{RUN_STEM}_judge_packet.md"
JUDGE_REVIEW_PATH = OUTPUT_DIR / f"{RUN_STEM}_judge.md"


def variant_paths(variant: str) -> dict[str, Path | str]:
    if variant not in VARIANT_STEMS:
        raise ValueError(f"Unknown variant: {variant}")
    stem = VARIANT_STEMS[variant]
    return {
        "variant": variant,
        "stem": stem,
        "code": OUTPUT_DIR / f"{stem}.py",
        "response": OUTPUT_DIR / f"{stem}.md",
        "check_log": OUTPUT_DIR / f"{stem}_checks.txt",
        "judge_packet": OUTPUT_DIR / f"{stem}_judge_packet.md",
        "judge_review": OUTPUT_DIR / f"{stem}_judge.md",
    }


def find_rules_path() -> Path:
    input_dir = Path("inputs")
    if not input_dir.exists():
        raise FileNotFoundError("Missing inputs/ directory")

    rules_paths = sorted(
        path
        for path in input_dir.iterdir()
        if path.stem == "game_rules" and path.suffix.lower() in {".txt", ".pdf"}
    )
    if not rules_paths:
        raise FileNotFoundError("Keep exactly one of inputs/game_rules.txt or inputs/game_rules.pdf")
    if len(rules_paths) > 1:
        names = ", ".join(path.name for path in rules_paths)
        raise RuntimeError(f"Multiple game_rules files found; keep exactly one: {names}")
    return rules_paths[0]


def read_rules_text(rules_path: Path) -> str:
    if rules_path.suffix.lower() == ".txt":
        return rules_path.read_text(encoding="utf-8")

    if rules_path.suffix.lower() == ".pdf":
        try:
            from pypdf import PdfReader
        except ImportError as exc:
            raise ImportError("PDF rulebooks require pypdf; install requirements.txt") from exc

        reader = PdfReader(str(rules_path))
        text = "\n\n".join(
            page_text.strip()
            for page in reader.pages
            for page_text in [page.extract_text() or ""]
            if page_text.strip()
        )
        if not text:
            raise ValueError(f"No extractable text found in {rules_path}")
        return text

    raise ValueError(f"Unsupported rules file type: {rules_path.suffix}")


def optional_generation_inputs() -> list[tuple[str, Path, str]]:
    items: list[tuple[str, Path, str]] = []
    if USE_OPEN_SPIEL_BACKBONE and BACKBONE_PATH.exists():
        items.append(("OpenSpiel backbone", BACKBONE_PATH, BACKBONE_PATH.read_text(encoding="utf-8")))
    if USE_IMPLEMENTATION_BRIEF and IMPLEMENTATION_BRIEF_PATH.exists():
        items.append(
            (
                "Implementation brief",
                IMPLEMENTATION_BRIEF_PATH,
                IMPLEMENTATION_BRIEF_PATH.read_text(encoding="utf-8"),
            )
        )
    return items


def get_pi_path() -> str:
    pi_path = shutil.which("pi") or shutil.which("pi.cmd")
    if pi_path is not None:
        return pi_path

    windows_fallback = Path.home() / "AppData/Roaming/npm/pi.cmd"
    if windows_fallback.exists():
        return str(windows_fallback)

    raise FileNotFoundError("Could not find pi or pi.cmd")


def extract_code_block(text: str) -> str | None:
    match = re.search(r"```python\s*(.*?)```", text, re.IGNORECASE | re.DOTALL)
    if match is None:
        return None
    return match.group(1).strip() + "\n"


def build_one_shot_prompt() -> str:
    if not PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing prompt file: {PROMPT_PATH}")

    rules_text = read_rules_text(find_rules_path())
    prompt_parts = [PROMPT_PATH.read_text(encoding="utf-8")]
    for label, path, text in optional_generation_inputs():
        prompt_parts.append(f"# {label}: {path.as_posix()}\n\n{text}")
    return "\n\n".join(prompt_parts) + "\n\nHier folgt die Spielanleitung:\n\n" + rules_text


def build_agentic_prompt() -> str:
    if not PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing prompt file: {PROMPT_PATH}")

    rules_path = find_rules_path()
    rules_text = read_rules_text(rules_path)
    lines = [
        "Work inside this repository with the available built-in pi tools only.",
        "",
        "Before you write the final file, read these inputs yourself:",
        f"- {PROMPT_PATH.as_posix()}",
        f"- {rules_path.as_posix()}",
    ]
    if USE_OPEN_SPIEL_BACKBONE and BACKBONE_PATH.exists():
        lines.append(f"- {BACKBONE_PATH.as_posix()}")
    if USE_IMPLEMENTATION_BRIEF and IMPLEMENTATION_BRIEF_PATH.exists():
        lines.append(f"- {IMPLEMENTATION_BRIEF_PATH.as_posix()}")
    lines += [
        "",
        "You may also inspect these checks to align the BoardBench API and action naming expectations:",
        "- checks/04_required_api.py",
        "- checks/05_random_rollouts.py",
        "- checks/99_openspiel_compare.py",
        "",
        f"Write the final Python module to {CODE_PATH.as_posix()}.",
        "Use only the rule text included below as the game source of truth.",
        "Do not use outside game knowledge or remembered rules.",
        "Keep the code to one self-contained standard-library Python file.",
        "",
        "Your final response must contain:",
        "1. Open questions / assumptions",
        "2. one fenced python code block with the exact final file content",
        "",
        "Rule text:",
        "",
        rules_text,
    ]
    return "\n".join(lines)


def pi_command() -> list[str]:
    base = [
        get_pi_path(),
        "-p",
        "--no-session",
        "--model",
        LLM_MODEL,
        "--no-extensions",
        "--no-skills",
        "--no-prompt-templates",
        "--no-context-files",
    ]
    if RUN_VARIANT == "oneshot":
        base.append("--no-tools")
    else:
        base += ["--tools", "read,write,edit,bash,grep,find,ls"]
    return base


def run_generation() -> subprocess.CompletedProcess[str]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    prompt_text = build_one_shot_prompt() if RUN_VARIANT == "oneshot" else build_agentic_prompt()
    command = pi_command()

    print(f"variant: {RUN_VARIANT}")
    print(f"game: {GAME}")
    print(f"model: {LLM_MODEL}")
    print(f"rules: {find_rules_path()}")
    print("command:", shlex.join(command))

    result = subprocess.run(
        command,
        input=prompt_text,
        capture_output=True,
        text=True,
        timeout=TIMEOUT_SECONDS,
    )

    RESPONSE_PATH.write_text(result.stdout or "", encoding="utf-8")
    if result.returncode != 0:
        stderr = (result.stderr or "").strip()
        stdout = (result.stdout or "").strip()
        raise RuntimeError(stderr or stdout or "pi call failed")

    code = extract_code_block(result.stdout or "")
    if code is not None:
        CODE_PATH.write_text(code, encoding="utf-8")
        print(f"Saved raw response: {RESPONSE_PATH}")
        print(f"Saved extracted code: {CODE_PATH}")
    elif CODE_PATH.exists():
        print(f"Saved raw response: {RESPONSE_PATH}")
        print(f"No fenced python block found; keeping file written by pi: {CODE_PATH}")
    else:
        raise RuntimeError("No fenced python block found in the LLM response and no code file was written")

    if (result.stderr or "").strip():
        print("\npi stderr:\n" + result.stderr.strip())
    return result


def check_command(
    code_path: Path,
    judge_path: Path,
    *,
    include_judge: bool = False,
    include_final: bool = False,
) -> list[str]:
    check_script = Path("checks/run_checks.py")
    if not check_script.exists():
        raise FileNotFoundError("Could not find checks/run_checks.py")

    cmd = [
        sys.executable,
        "-u",
        str(check_script),
        "--game",
        GAME,
        "--code-path",
        str(code_path),
        "--rollouts",
        str(ROLLOUTS),
        "--max-steps",
        str(MAX_STEPS),
        "--seed",
        str(CHECK_SEED),
    ]
    if include_judge:
        cmd.append("--include-judge")
        cmd += ["--judge-path", str(judge_path)]
    if include_final:
        cmd.append("--include-final")
    return cmd


def run_checks_for(
    *,
    code_path: Path,
    judge_path: Path,
    include_judge: bool = False,
    include_final: bool = False,
    log_path: Path | None = None,
    label: str | None = None,
) -> subprocess.CompletedProcess[str]:
    cmd = check_command(
        code_path=code_path,
        judge_path=judge_path,
        include_judge=include_judge,
        include_final=include_final,
    )
    tag = label or code_path.stem
    print(f"running checks for {tag}: {code_path}")
    print("command:", shlex.join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")

    if log_path is not None:
        log_path.write_text(output, encoding="utf-8")
        print(f"Saved check log: {log_path}")

    print(output)
    if result.returncode != 0:
        print(f"checks failed with exit code {result.returncode}")
    return result



def run_pair_action_compare(
    *,
    left_code_path: Path,
    right_code_path: Path,
    left_label: str = "oneshot",
    right_label: str = "agentic",
) -> subprocess.CompletedProcess[str]:
    compare_script = Path("checks/compare_pair.py")
    if not compare_script.exists():
        raise FileNotFoundError("Could not find checks/compare_pair.py")

    cmd = [
        sys.executable,
        "-u",
        str(compare_script),
        "--game",
        GAME,
        "--left-code-path",
        str(left_code_path),
        "--right-code-path",
        str(right_code_path),
        "--left-label",
        left_label,
        "--right-label",
        right_label,
        "--rollouts",
        str(ROLLOUTS),
        "--max-steps",
        str(MAX_STEPS),
        "--seed",
        str(CHECK_SEED),
    ]
    print('\n' + "=" * 80)
    print("running pair action-language comparison")
    print("command:", shlex.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    pair_log_path = OUTPUT_DIR / f"{GAME}_pair_action_compare.txt"
    pair_log_path.write_text(output, encoding="utf-8")
    print(f"Saved pair comparison log: {pair_log_path}")
    print(output)
    if result.returncode != 0:
        print(f"pair action-language comparison failed with exit code {result.returncode}")
    return result


def build_judge_packet_for(
    *,
    code_path: Path,
    check_log_path: Path,
    output_path: Path,
    judge_review_path: Path,
    variant: str,
) -> Path:
    if not code_path.exists():
        raise FileNotFoundError(f"Missing generated code: {code_path}")
    if not PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing prompt file: {PROMPT_PATH}")
    if not LLM_JUDGE_PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing judge prompt file: {LLM_JUDGE_PROMPT_PATH}")

    rules_path = find_rules_path()
    sections = [
        "# BoardBench judge packet",
        f"- game: {GAME}",
        f"- variant: {variant}",
        f"- generated code: {code_path.as_posix()}",
        f"- expected judge reply path: {judge_review_path.as_posix()}",
        "",
        "## Judge prompt",
        LLM_JUDGE_PROMPT_PATH.read_text(encoding="utf-8"),
        "",
        f"## Generation prompt ({PROMPT_PATH.as_posix()})",
        PROMPT_PATH.read_text(encoding="utf-8"),
    ]

    for label, path, text in optional_generation_inputs():
        sections += ["", f"## {label} ({path.as_posix()})", text]

    sections += [
        "",
        f"## Rule text ({rules_path.as_posix()})",
        read_rules_text(rules_path),
        "",
        f"## Generated code ({code_path.as_posix()})",
        "```python\n" + code_path.read_text(encoding="utf-8") + "```",
    ]

    if check_log_path.exists():
        sections += [
            "",
            f"## Check output ({check_log_path.as_posix()})",
            "```text\n" + check_log_path.read_text(encoding="utf-8") + "\n```",
        ]
    else:
        sections += [
            "",
            "## Check output",
            "_Run the checks cell first if you want to include deterministic check output._",
        ]

    output_path.write_text("\n\n".join(sections), encoding="utf-8")
    print(f"Saved judge packet: {output_path}")
    print(f"Save the judge reply as: {judge_review_path}")
    return output_path


SUMMARY_RE = re.compile(r"summary:\s+(\d+)/(\d+) checks, (\d+)/(\d+) units, ([0-9.]+)s")


def extract_summary(text: str) -> dict[str, int | float] | None:
    match = SUMMARY_RE.search(text)
    if match is None:
        return None
    return {
        "checks_passed": int(match.group(1)),
        "checks_total": int(match.group(2)),
        "units_passed": int(match.group(3)),
        "units_total": int(match.group(4)),
        "seconds": float(match.group(5)),
    }


print(f"Notebook ready for {RUN_VARIANT}: {CODE_PATH}")


## One-shot generation

This notebook is the strict one-prompt comparison baseline:

- `pi -p`
- no tools
- no extensions, skills, prompt templates, or context files
- prompt/backbone/brief/rules are inlined before the call

Outputs for the current `GAME`:

- `outputs/<game>_oneshot.md`
- `outputs/<game>_oneshot.py`


In [ ]:
run_generation()


## Checks for the current variant

Run this after generation.

- leave both toggles on `False` for the normal deterministic checks
- set `INCLUDE_LLM_JUDGE = True` after you saved the manual judge reply
- set `INCLUDE_OPENSPIEL_COMPARE = True` when you want the optional OpenSpiel comparison


In [ ]:
INCLUDE_LLM_JUDGE = False
INCLUDE_OPENSPIEL_COMPARE = False

run_checks_for(
    code_path=CODE_PATH,
    judge_path=JUDGE_REVIEW_PATH,
    include_judge=INCLUDE_LLM_JUDGE,
    include_final=INCLUDE_OPENSPIEL_COMPARE,
    log_path=CHECK_LOG_PATH,
    label=RUN_VARIANT,
)


## Manual LLM judge preparation

Run the checks cell first. Then this cell writes a ready-to-paste judge packet with:

- the judge prompt
- the generation prompt and optional extra context
- the rule text
- the generated code
- the latest deterministic check output

Use that packet with your chosen judge model and save the reply to the printed `..._judge.md` path.


In [ ]:
build_judge_packet_for(
    code_path=CODE_PATH,
    check_log_path=CHECK_LOG_PATH,
    output_path=JUDGE_PACKET_PATH,
    judge_review_path=JUDGE_REVIEW_PATH,
    variant=RUN_VARIANT,
)


## Run both variants with the same checks

Use this after both `outputs/<game>_oneshot.py` and `outputs/<game>_agentic.py` exist.

- keep both toggles on `False` for the normal side-by-side check run
- enable the judge toggle only after both judge replies were saved
- enable the OpenSpiel toggle when you want the optional reference comparison for both variants
- the pair action-language comparison normalizes emitted action names only; it does not add missing actions


In [ ]:

PAIR_INCLUDE_LLM_JUDGE = False
PAIR_INCLUDE_OPENSPIEL_COMPARE = False
PAIR_COMPARE_ACTION_LANGUAGE = True

pair_results = {}
for variant in ["oneshot", "agentic"]:
    paths = variant_paths(variant)
    if not Path(paths["code"]).exists():
        print(f"Missing {variant} code: {paths['code']}")
        continue

    print("\n" + "=" * 80)
    result = run_checks_for(
        code_path=Path(paths["code"]),
        judge_path=Path(paths["judge_review"]),
        include_judge=PAIR_INCLUDE_LLM_JUDGE,
        include_final=PAIR_INCLUDE_OPENSPIEL_COMPARE,
        log_path=Path(paths["check_log"]),
        label=variant,
    )
    pair_results[variant] = {
        "returncode": result.returncode,
        "summary": extract_summary((result.stdout or "") + (result.stderr or "")),
    }

if PAIR_COMPARE_ACTION_LANGUAGE and all(Path(variant_paths(variant)["code"]).exists() for variant in ["oneshot", "agentic"]):
    pair_compare_result = run_pair_action_compare(
        left_code_path=Path(variant_paths("oneshot")["code"]),
        right_code_path=Path(variant_paths("agentic")["code"]),
        left_label="oneshot",
        right_label="agentic",
    )
    pair_results["pair_action_compare"] = {
        "returncode": pair_compare_result.returncode,
        "summary": extract_summary((pair_compare_result.stdout or "") + (pair_compare_result.stderr or "")),
    }

print("\npair summary:")
for variant in ["oneshot", "agentic", "pair_action_compare"]:
    info = pair_results.get(variant)
    if info is None:
        print(f"- {variant}: not run")
        continue

    summary = info["summary"]
    if summary is None:
        print(f"- {variant}: returncode={info['returncode']} (no summary parsed)")
        continue

    print(
        f"- {variant}: {summary['checks_passed']}/{summary['checks_total']} checks, "
        f"{summary['units_passed']}/{summary['units_total']} units, "
        f"{summary['seconds']:.2f}s, returncode={info['returncode']}"
    )

if not all(variant in pair_results for variant in ["oneshot", "agentic"]):
    print("\nGenerate both variants first for a real side-by-side comparison.")
